## Assignment 1: Supervised Machine Learning

Welcome to the first assignment of CS 541! This assignment prepares you with some useful tools that are widely used in NLP. This assignment must be done individually.

After this assignment, you should be able to:  
1. Load a dataset from huggingface's dataset library, and do some exploratory analyses.  
2. Use scikit-learn to build and train a feature-based model.  
3. Use pytorch to build and train a feature-based model.  
4. Use Optuna to automatically search for hyperparameters.  

In CS541, any work generated by an AI shouldn't be included without declaration. If you include material generated by an AI, the level of AI use should be properly documented, and the actual tool should be noted (e.g., "I used Codex to proofread the codes and draft the analysis"). 

**AI declaration (for the reference solution):** This reference solution was drafted with the help of Claude (Anthropic) and reviewed by the TA. Students must include their own declaration in their submission; a missing declaration is a policy violation (see rubric at the end).

> **Note on the starter code.** Every cell that was provided in the starter notebook is kept **verbatim** below (same imports, same function names, same signatures, same `random_state`/seed calls). Only the `TODO` bodies are filled in, and helper functions are added *next to* the starter functions rather than by changing them. Students were not allowed to change the provided signatures either — see the rubric.

**Runtime.** `X_train` is a dense 67 349 × 3 120 matrix (≈ 1.7 GB in float64), and the provided collate function converts lists of NumPy rows to tensors, which PyTorch itself flags as slow. Expect the sklearn grid to take a few minutes per configuration and every PyTorch epoch a few minutes on CPU; the grids and the Optuna budget are kept small for that reason. All reported accuracies in this notebook come from actually running the cells (`Kernel → Restart & Run All`) — the TA copy must be run once so the numbers in the tables are filled in before it is used as the reference.

### 1. Load the dataset (5')
First, we are going to load the datasets from huggingface's `datasets` library.
Do some exploratory analysis on the dataset.  
1.1 Print out one example in the dataset. Briefly comment on what it contains.  
1.2 For each of the train, validation, and test set, compute the following statistics: 
- The number of data samples with each class label.  
- The mean and std of the sentence lengths (in words) of each `question`.  

1.3 Vectorize the validation set of the dataset, following the approaches specified in the train set example.

In [ ]:
import pandas as pd 
import numpy as np 
from datasets import load_dataset  # huggingface datasets

ds = load_dataset("stanfordnlp/sst2")

Reusing dataset glue (/Users/zhuzi/.cache/huggingface/datasets/glue/sst2/1.0.0/dacbe3125aa31d7f70367a07a8a9e72a5a0bfeb5fc42e75c9db75b96da6053ad)


  0%|          | 0/3 [00:00<?, ?it/s]

In [2]:
ds

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

#### 1.1 One example from the dataset

In [ ]:
# TODO -- Print out one example in the dataset. Briefly comment on what it contains.  
example = ds["train"][0]
print(example)
print()
print("Fields:", list(example.keys()))
print("Label meaning:", ds["train"].features["label"])

**Comment.** Each example has three fields:
- `sentence` — a (lower-cased, tokenised) movie-review sentence/phrase from the Stanford Sentiment Treebank, e.g. *"hide new secretions from the parental units"*;
- `label` — the binary sentiment, `0 = negative`, `1 = positive` (the `ClassLabel` feature gives the names);
- `idx` — the integer row index of the example inside its split.

So SST-2 is a binary sentence-level sentiment-classification task.

In [3]:
train_data = pd.DataFrame(ds["train"])
X_train_text = train_data["sentence"]
Y_train = train_data["label"]

val_data = pd.DataFrame(ds["validation"])
X_val_text = val_data["sentence"]
Y_val = val_data["label"]

test_data = pd.DataFrame(ds["test"])
X_test_text = test_data["sentence"]
Y_test = test_data["label"]

#### 1.2 Exploratory statistics (class counts, sentence-length mean / std)

In [ ]:
# TODO -- compute the exploratory statistics
rows = []
for name, df in [("train", train_data), ("validation", val_data), ("test", test_data)]:
    lengths = df["sentence"].str.split().str.len()          # sentence length in words
    counts = df["label"].value_counts().sort_index()
    print(f"===== {name} ({len(df)} samples) =====")
    print("Class counts:")
    print(counts.to_string())
    print(f"Sentence length (words): mean = {lengths.mean():.2f}, "
          f"std = {lengths.std(ddof=0):.2f} (population std), "
          f"{lengths.std(ddof=1):.2f} (sample std)")
    print()
    rows.append({"split": name, "n": len(df),
                 **{f"label={k}": v for k, v in counts.items()},
                 "len_mean": lengths.mean(), "len_std_pop": lengths.std(ddof=0), "len_std_sample": lengths.std(ddof=1)})

stats = pd.DataFrame(rows).set_index("split")
stats

**Comments.**
- The assignment text says the length of each `question`; the field in SST-2 is called `sentence`, so the statistic is computed on `sentence`.
- Train is mildly imbalanced (≈ 56 % positive / 44 % negative); validation is close to balanced.
- **The GLUE test split has hidden labels — every test label is `-1`.** A correct answer *must* notice this: the test class counts are simply `-1: 1821`, and the test set cannot be used to measure accuracy anywhere in this assignment. That is why all model selection and reporting below is done on the validation set.
- Train sentences are much shorter on average than validation/test sentences (the train split contains many sub-phrases of the parse trees, the dev/test splits contain complete sentences).
- `pandas.Series.std()` uses the sample std (`ddof=1`) whereas `numpy.std()` uses the population std (`ddof=0`); either is acceptable if the student is consistent, so both are printed.

Next we are going to vectorize the texts using TfidfVectorizer, then compute the Tf-idf features.   

In [47]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer 
from sklearn.neural_network import MLPClassifier 

counter = CountVectorizer(min_df=10, max_df=20) 
counter.fit(X_train_text)
print("Vocabulary size:", len(counter.vocabulary_))
X_train_counts = counter.transform(X_train_text)
print(X_train_counts.shape) 
count2tfidf = TfidfTransformer(use_idf=True).fit(X_train_counts)
X_train = count2tfidf.transform(X_train_counts).toarray()
print(X_train.shape)

Vocabulary size: 3120
(67349, 3120)
(67349, 3120)


#### 1.3 Vectorise the validation set with the *train-fitted* vectorizer

In [ ]:
# TODO - Use the counter to convert X_val_text to occurrence vectors
# Note: don't create a new CountVectorizer, as we want to compute the vocabulary only on the train set
X_val_counts = counter.transform(X_val_text)

# TODO - use count2tfidf to transform the counts into Tfidf features
# Note: don't create a new TfidfTransformer
X_val = count2tfidf.transform(X_val_counts).toarray()

print(X_val_counts.shape, X_val.shape)
print("Validation rows with an all-zero feature vector:", int((X_val_counts.sum(axis=1) == 0).sum()),
      "of", X_val.shape[0])

**Comment.** Only `transform` is called — the vocabulary (`counter.vocabulary_`) and the IDF weights (`count2tfidf.idf_`) were learned on the training split and must be reused, otherwise the validation features would live in a different space and the trained model could not be applied to them. Note that with `min_df=10, max_df=20` the vocabulary only contains fairly rare words (3 120 of them), so a large fraction of validation sentences map to the all-zero vector — this caps the accuracy that any model in this assignment can reach. Students are **not** expected to change `CountVectorizer(min_df=10, max_df=20)`; it is provided code.

### 2. Train scikit-learn models (10')
Train a two-layer MLPClassifier using `random_state=0`. Manually tune the hyperparameters on the validation set. Report the procedure of hyperparameter tuning. Specifically: report the hyperparameters you have tried, and their results.  

After you are satisfied with the validation set performances, report the validation set performance. Use this set of hyperparameters and repeat the model training procedure for five times using `random_state` as 1, 2, 3, 31, 42 respectively. Record the five accuracy numbers.

**Design.** *Two-layer MLP* is read as two `Linear` layers, i.e. `hidden_layer_sizes=(h,)` — one hidden layer plus the output layer, matching the `MLP(all_layer_sizes)` class in Section 3 where `all_layer_sizes = [3120, h, 2]`. (A student who uses `hidden_layer_sizes=(h1, h2)` and *says* they interpret "two-layer" as two hidden layers is not penalised, as long as they are consistent in Sections 2 and 3.)

The starter function `train_sklearn_model(X_train, Y_train, X_val, Y_val)` keeps its exact signature and its exact call line. Everything Section 2 asks for happens inside it, in order: (1) manual tuning grid with `random_state=0`, printed as a table; (2) the chosen configuration re-trained with `random_state=0` and its validation accuracy reported; (3) the same configuration re-trained with `random_state` = 1, 2, 3, 31, 42 and the five accuracies recorded. The results are also stored in module-level variables (`SK_BEST`, `SK_SEED_ACCS`) so the bonus section can reuse them.

In [ ]:
# Starter
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

SK_RESULTS_DF, SK_BEST, SK_SEED_ACCS = None, None, None   # filled in by train_sklearn_model

def train_sklearn_model(X_train, Y_train, X_val, Y_val):
    global SK_RESULTS_DF, SK_BEST, SK_SEED_ACCS

    def fit_eval(hp, random_state):
        """Train one two-layer MLPClassifier with hyper-parameters hp; return validation accuracy."""
        clf = MLPClassifier(hidden_layer_sizes=hp["hidden_layer_sizes"],
                            activation="relu", solver="adam",
                            learning_rate_init=hp["learning_rate_init"],
                            alpha=hp["alpha"],
                            batch_size=hp["batch_size"],
                            max_iter=hp["max_iter"],
                            random_state=random_state)
        clf.fit(X_train, Y_train)
        return clf.score(X_val, Y_val)

    # ---- (1) manual hyper-parameter tuning on the validation set, random_state=0 ----
    grid = [
        dict(hidden_layer_sizes=(64,),  learning_rate_init=1e-3, alpha=1e-4, batch_size=128, max_iter=15),
        dict(hidden_layer_sizes=(128,), learning_rate_init=1e-3, alpha=1e-4, batch_size=128, max_iter=15),
        dict(hidden_layer_sizes=(256,), learning_rate_init=1e-3, alpha=1e-4, batch_size=128, max_iter=15),
        dict(hidden_layer_sizes=(128,), learning_rate_init=1e-2, alpha=1e-4, batch_size=128, max_iter=15),
        dict(hidden_layer_sizes=(128,), learning_rate_init=1e-4, alpha=1e-4, batch_size=128, max_iter=15),
        dict(hidden_layer_sizes=(128,), learning_rate_init=1e-3, alpha=1e-2, batch_size=128, max_iter=15),
        dict(hidden_layer_sizes=(256,), learning_rate_init=1e-3, alpha=1e-2, batch_size=128, max_iter=15),
        dict(hidden_layer_sizes=(128,), learning_rate_init=1e-3, alpha=1e-4, batch_size=32,  max_iter=15),
    ]
    print("=== Manual tuning (random_state=0) ===")
    results = []
    for i, hp in enumerate(grid, 1):
        acc = fit_eval(hp, random_state=0)
        print(f"[{i}/{len(grid)}] {hp} -> val acc {acc:.4f}")
        results.append({**hp, "val_acc": acc})
    SK_RESULTS_DF = pd.DataFrame(results)
    print("\nTuning results table:")
    print(SK_RESULTS_DF.to_string())

    # ---- (2) final model with the chosen hyper-parameters, random_state=0 ----
    best_row = SK_RESULTS_DF.loc[SK_RESULTS_DF["val_acc"].idxmax()]
    SK_BEST = dict(hidden_layer_sizes=tuple(best_row["hidden_layer_sizes"]),
                   learning_rate_init=float(best_row["learning_rate_init"]),
                   alpha=float(best_row["alpha"]),
                   batch_size=int(best_row["batch_size"]),
                   max_iter=int(best_row["max_iter"]))
    final_acc = fit_eval(SK_BEST, random_state=0)
    print("\nChosen hyper-parameters:", SK_BEST)
    print("Validation accuracy with random_state=0: {:.4f}".format(final_acc))

    # ---- (3) repeat with random_state = 1, 2, 3, 31, 42 ----
    print("\n=== Five re-runs with the chosen hyper-parameters ===")
    SK_SEED_ACCS = []
    for s in [1, 2, 3, 31, 42]:
        acc = fit_eval(SK_BEST, random_state=s)
        SK_SEED_ACCS.append(acc)
        print(f"random_state={s:>2d}: val acc = {acc:.4f}")
    print("five accuracies:", [round(a, 4) for a in SK_SEED_ACCS])
    print("mean = {:.4f}, std = {:.4f}".format(np.mean(SK_SEED_ACCS), np.std(SK_SEED_ACCS, ddof=1)))

    return {"val_acc_random_state_0": final_acc, "five_seed_accs": SK_SEED_ACCS}

train_sklearn_model(X_train, Y_train, X_val, Y_val)

**Tuning report.** The printed table is the report: every configuration tried with its validation accuracy, the chosen configuration (highest validation accuracy), its `random_state=0` accuracy, and the five seed accuracies with mean ± std. The spread across seeds is the "noise floor": two settings whose validation accuracies differ by less than roughly one std should not be called *different* — this is what Section 5 tests formally.

### 3. Train a pytorch model (10')
Here you will repeat the training of a two-layer fully-connected neural network using pytorch. Following are some specifications that may be helpful:  
- For each of the train and validation set, specify a dataloader, preferrably using `torch.utils.data.DataLoader`.  
- Use an optimizer of your choice. Adam, AdamW and SGD are popular choices.  
- Designate a number, `train_epochs`, as the number of passes through the dataset during training. Each pass through the training dataset is called an epoch.  
  - During the epoch, there may be many steps. In each step, load a batch of data from the dataloader. Compute the loss. Do a `backward()` pass to compute the gradients. Call a `step()` from the optimizer to update the model's parameters. Then zero out the gradients.
- At the end of each epoch, go through a validation run. Do *not* optimize the model during the validation run. Compute the accuracy of the model on this validation run, and print it out.

Tune the hyperparameters on the validation set. Report the hyperparameters you have tried, and their results. 

After you are satisfied with the validation set performances, record the set of hyperparameters. Use this set of hyperparameters, and repeat the model training procedure for five times using 1, 2, 3, 31, 42 as random seeds respectively. You can use `torch.manual_seed()` to set the random seeds. Record the five accuracy numbers.

**Design.** `MLP`, `my_collate_function`, `prepare_zipped_XY` and `train_pytorch_model` keep their exact starter signatures, and the starter call `train_pytorch_model(X_train, Y_train, X_val, Y_val)` is kept verbatim. Because the function takes no seed or hyper-parameter arguments, the *body* reads them from module-level settings (`SEED`, `TRAIN_EPOCHS`, `BATCH_SIZE`, `LEARNING_RATE`, `HIDDEN_SIZES`) — the lines the starter marked `torch.manual_seed(1)` / `train_epochs = None` etc. This is the minimal change that allows the assignment's required tuning loop and seed loop (1, 2, 3, 31, 42) without touching the signature.

Implementation notes:
- `MLP.net` is an `nn.Sequential` built from an `OrderedDict` (the starter imports `OrderedDict` for this purpose): `Linear → ReLU → Linear`. `all_layer_sizes = [3120, hidden, 2]` gives the two-layer network.
- `forward` returns **logits**; `nn.CrossEntropyLoss` applies the softmax internally, so no `Softmax` layer is added (adding one *and* using `CrossEntropyLoss` is a common mistake).
- Order of operations in each step: `loss.backward()` → `optim.step()` → `optim.zero_grad()` (zeroing before `backward()` is also fine; zeroing between `backward()` and `step()` is a bug).
- The validation pass uses `model.eval()` and `torch.no_grad()`; no optimiser step is taken.
- `X` is cast to `float32` before zipping to halve memory; `torch.tensor(list_of_numpy_arrays)` in the provided collate function is slow (PyTorch warns about it) — that is starter code and left untouched, which is why the grids below are kept small.

In [ ]:
# Starter
import torch
import torch.nn as nn
from collections import OrderedDict
from torch.utils.data import DataLoader

# Module-level settings read by train_pytorch_model (the function signature stays as provided).
# These defaults are the configuration chosen after the manual tuning in 3.1.
SEED = 1
TRAIN_EPOCHS = 5
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
HIDDEN_SIZES = (128,)
VERBOSE = True          # per-epoch printing (switched off only inside the tuning / seed loops to keep output short)

class MLP(nn.Module):
    def __init__(self, all_layer_sizes):
        super().__init__()
        layers = OrderedDict()
        n_linear = len(all_layer_sizes) - 1
        for i in range(n_linear):
            layers[f"linear{i+1}"] = nn.Linear(all_layer_sizes[i], all_layer_sizes[i+1])
            if i < n_linear - 1:                 # no activation after the output layer (logits)
                layers[f"relu{i+1}"] = nn.ReLU()
        self.net = nn.Sequential(layers)

    def forward(self, X):
        return self.net(X)                       # logits, shape (batch, n_classes)

def my_collate_function(batch):
    batch_X, batch_Y = [], []
    for item in batch:
        batch_X.append(item[0])
        batch_Y.append(item[1])
    return torch.tensor(batch_X).float(), torch.tensor(batch_Y).long()

def prepare_zipped_XY(X, Y):
    zipped = []
    for i in range(len(X)):
        zipped.append((X[i], Y[i]))
    return zipped

def train_pytorch_model(X_train, Y_train, X_val, Y_val):
    # Define the manual seed
    torch.manual_seed(SEED)

    # Define the hyperparameters
    train_epochs = TRAIN_EPOCHS
    batch_size = BATCH_SIZE
    learning_rate = LEARNING_RATE
    hidden_sizes = HIDDEN_SIZES

    # Set up the model, optimizer, and dataloader
    n_features = X_train.shape[1]
    n_classes = int(len(np.unique(Y_train)))
    model = MLP([n_features, *hidden_sizes, n_classes])
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    train_zipped = prepare_zipped_XY(np.asarray(X_train, dtype=np.float32), np.asarray(Y_train))
    val_zipped   = prepare_zipped_XY(np.asarray(X_val,   dtype=np.float32), np.asarray(Y_val))
    train_dataloader = DataLoader(train_zipped, batch_size=batch_size, shuffle=True,  collate_fn=my_collate_function)
    val_dataloader   = DataLoader(val_zipped,   batch_size=256,        shuffle=False, collate_fn=my_collate_function)
    if VERBOSE:
        print ("Start training!")
    last_epoch_dev_acc = 0
    for epoch in range(train_epochs):
        model.train()
        for batch_X, batch_Y in train_dataloader:
            logits = model(batch_X)                # forward pass
            loss = loss_fn(logits, batch_Y)        # compute the loss
            loss.backward()                        # compute gradients
            optim.step()                           # update parameters
            optim.zero_grad()                      # zero out the gradients

        n_correct, n_total = 0, 0
        # End-of-epoch evaluation: no optimisation, no gradient tracking
        model.eval()
        with torch.no_grad():
            for batch_X, batch_Y in val_dataloader:
                preds = model(batch_X).argmax(dim=1)
                n_correct += (preds == batch_Y).sum().item()
                n_total += batch_Y.shape[0]

        last_epoch_dev_acc = n_correct/n_total 
        if VERBOSE:
            print("Epoch {}, val accuracy {:.2f}".format(epoch+1, last_epoch_dev_acc))

    return last_epoch_dev_acc

train_pytorch_model(X_train, Y_train, X_val, Y_val)

#### 3.1 Manual hyper-parameter tuning on the validation set (seed 1)

In [ ]:
pt_grid = [
    dict(TRAIN_EPOCHS=5, BATCH_SIZE=128, LEARNING_RATE=1e-3, HIDDEN_SIZES=(128,)),
    dict(TRAIN_EPOCHS=5, BATCH_SIZE=128, LEARNING_RATE=1e-2, HIDDEN_SIZES=(128,)),
    dict(TRAIN_EPOCHS=5, BATCH_SIZE=128, LEARNING_RATE=1e-4, HIDDEN_SIZES=(128,)),
    dict(TRAIN_EPOCHS=5, BATCH_SIZE=128, LEARNING_RATE=1e-3, HIDDEN_SIZES=(256,)),
    dict(TRAIN_EPOCHS=5, BATCH_SIZE=32,  LEARNING_RATE=1e-3, HIDDEN_SIZES=(128,)),
    dict(TRAIN_EPOCHS=8, BATCH_SIZE=128, LEARNING_RATE=1e-3, HIDDEN_SIZES=(64,)),
]

SEED, VERBOSE = 1, False
pt_results = []
for i, hp in enumerate(pt_grid, 1):
    globals().update(hp)                                   # set TRAIN_EPOCHS, BATCH_SIZE, LEARNING_RATE, HIDDEN_SIZES
    acc = train_pytorch_model(X_train, Y_train, X_val, Y_val)
    print(f"[{i}/{len(pt_grid)}] {hp} -> val acc {acc:.4f}")
    pt_results.append({**hp, "val_acc": acc})
VERBOSE = True

pt_results_df = pd.DataFrame(pt_results)
best_pt_row = pt_results_df.loc[pt_results_df["val_acc"].idxmax()]
PT_BEST = dict(TRAIN_EPOCHS=int(best_pt_row["TRAIN_EPOCHS"]), BATCH_SIZE=int(best_pt_row["BATCH_SIZE"]),
               LEARNING_RATE=float(best_pt_row["LEARNING_RATE"]), HIDDEN_SIZES=tuple(best_pt_row["HIDDEN_SIZES"]))
globals().update(PT_BEST)                                  # keep the chosen configuration active from here on
print("\nChosen pytorch hyper-parameters:", PT_BEST)
print("Validation accuracy (seed 1): {:.4f}".format(best_pt_row["val_acc"]))
pt_results_df

#### 3.2 Five runs with seeds 1, 2, 3, 31, 42 using the chosen hyper-parameters

In [ ]:
SEEDS = [1, 2, 3, 31, 42]
PT_SEED_ACCS = []
VERBOSE = False
for SEED in SEEDS:                                         # train_pytorch_model reads SEED -> torch.manual_seed(SEED)
    acc = train_pytorch_model(X_train, Y_train, X_val, Y_val)
    PT_SEED_ACCS.append(acc)
    print(f"seed={SEED:>2d}: val acc = {acc:.4f}")
SEED, VERBOSE = 1, True

print("\npytorch five-seed accuracies:", [round(a, 4) for a in PT_SEED_ACCS])
print("mean = {:.4f}, std = {:.4f}".format(np.mean(PT_SEED_ACCS), np.std(PT_SEED_ACCS, ddof=1)))

**Required in the write-up:** the table of configurations tried (`pt_results_df`), the chosen configuration, the per-epoch validation accuracies printed by `train_pytorch_model`, and the five seed accuracies with mean ± std.

### 4. Hyperparameter tuning (10')
This question requires modifying your previous pytorch training scripts. Use Optuna to find the hyperparameters that can maximize the accuracy on the validation set.  

The range of hyperparameters don't need to be too large (i.e., the total program should still be runnable within a reasonable time). The most important hyperparameter is the learning rate. Other hyperparameters that you can tune include the train epochs, batch size, hidden sizes, etc.  

When you are satisfied with the hyperparameters, report the hyperparameter and the resulting validation accuracy.

**Design / the one thing students most often get wrong.** The starter creates the study with `optuna.create_study()` and that line is *provided code*. Optuna's default `direction` is **`minimize`**, so the objective must return something to be minimised — here the **validation error rate `1 − accuracy`**. Returning the accuracy directly with the default study makes Optuna search for the *worst* hyper-parameters. (Passing `direction="maximize"` and returning accuracy is an equally correct alternative; returning accuracy with the default study is a bug and is penalised.)

As the starter instructs, `train_pytorch_model_with_optuna` is `train_pytorch_model` with the hyper-parameters replaced by `trial.suggest_*` calls (seed fixed to 1 so trials are comparable). Both starter signatures and the starter call `find_optimal_hyper_params(X_train, Y_train, X_val, Y_val)` are unchanged; the function prints the best trial and the full trial table.

Search space (kept small so 20 trials finish in reasonable time):
- `learning_rate` — log-uniform in [1e-4, 1e-1] (the most important hyper-parameter, hence log scale);
- `batch_size` — categorical {64, 128, 256};
- `hidden_size` — integer in [64, 512] in steps of 64;
- `train_epochs` — integer in [2, 5].

In [ ]:
# Starter
import optuna 

def train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val):
    # Same procedure as train_pytorch_model, but the hyperparameters are recommended by the Optuna trial
    torch.manual_seed(1)

    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [64, 128, 256])
    hidden_size   = trial.suggest_int("hidden_size", 64, 512, step=64)
    train_epochs  = trial.suggest_int("train_epochs", 2, 5)
    hidden_sizes  = (hidden_size,)

    n_features = X_train.shape[1]
    n_classes = int(len(np.unique(Y_train)))
    model = MLP([n_features, *hidden_sizes, n_classes])
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    train_zipped = prepare_zipped_XY(np.asarray(X_train, dtype=np.float32), np.asarray(Y_train))
    val_zipped   = prepare_zipped_XY(np.asarray(X_val,   dtype=np.float32), np.asarray(Y_val))
    train_dataloader = DataLoader(train_zipped, batch_size=batch_size, shuffle=True,  collate_fn=my_collate_function)
    val_dataloader   = DataLoader(val_zipped,   batch_size=256,        shuffle=False, collate_fn=my_collate_function)

    last_epoch_dev_acc = 0
    for epoch in range(train_epochs):
        model.train()
        for batch_X, batch_Y in train_dataloader:
            loss = loss_fn(model(batch_X), batch_Y)
            loss.backward()
            optim.step()
            optim.zero_grad()

        n_correct, n_total = 0, 0
        model.eval()
        with torch.no_grad():
            for batch_X, batch_Y in val_dataloader:
                preds = model(batch_X).argmax(dim=1)
                n_correct += (preds == batch_Y).sum().item()
                n_total += batch_Y.shape[0]
        last_epoch_dev_acc = n_correct/n_total

    trial.set_user_attr("val_acc", last_epoch_dev_acc)
    # optuna.create_study() minimises by default -> return the validation ERROR rate
    return 1.0 - last_epoch_dev_acc

def find_optimal_hyper_params(X_train, Y_train, X_val, Y_val):
    # Start an Optuna study
    study = optuna.create_study()

    # objective takes only the trial; the data are captured through a closure
    def objective(trial):
        return train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val)
    study.optimize(objective, n_trials=20)

    best = study.best_trial
    print("Best trial #{}".format(best.number))
    print("Best hyper-parameters:", best.params)
    print("Best validation accuracy: {:.4f}  (objective value = error rate {:.4f})".format(
        1.0 - best.value, best.value))
    print("\nAll trials, sorted by validation accuracy (highest first):")
    trials_df = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs"))
    print(trials_df.rename(columns={"value": "val_error"}).sort_values("val_error").to_string())

find_optimal_hyper_params(X_train, Y_train, X_val, Y_val)

**Required in the write-up:** the search space, the best hyper-parameters (`study.best_params`) and the resulting validation accuracy (`1 − study.best_value`). Because Optuna's result is itself selected on the validation set, it is an optimistically biased estimate — a good answer notes that a truly held-out set would be needed to quote a final number (the SST-2 test labels are hidden, so none is available here).

### 5. Bonus: Compare the performances of the two methods (2')
Use an appropriate $t$ test, compare the five performance numbers of the sklearn model and the pytorch model *under the same set of hyperparameters*. Do their results differ?

Note: The scores for bonus will be added to the A1 total score, but the total score will be capped to 100%.

**Which test?** Two sets of five accuracies, obtained with the *same* hyper-parameters (hidden size, learning rate, batch size, number of epochs, Adam) in sklearn and in PyTorch. Although the same seed values (1, 2, 3, 31, 42) are used, a seed means something different in the two libraries (different RNGs, different initialisation and shuffling), so the two samples are **not naturally paired** — the appropriate test is an **independent two-sample t-test**. Welch's version (`equal_var=False`) is used because there is no reason to assume equal variances. A paired t-test is accepted only if the student explicitly argues that seed *k* in both libraries is a matched pair (it is a weaker argument, but not wrong).

To make the comparison fair, the sklearn model is re-trained with exactly the PyTorch hyper-parameters (`hidden_layer_sizes=HIDDEN_SIZES`, `learning_rate_init=LEARNING_RATE`, `batch_size=BATCH_SIZE`, `max_iter=TRAIN_EPOCHS`, solver `adam`, `alpha=0` because the PyTorch model has no weight decay).

In [ ]:
from scipy import stats

# Re-train the sklearn model under the SAME hyper-parameters as the chosen PyTorch configuration
sk_same_accs = []
for s in SEEDS:
    clf = MLPClassifier(hidden_layer_sizes=HIDDEN_SIZES, activation="relu", solver="adam",
                        learning_rate_init=LEARNING_RATE, batch_size=BATCH_SIZE,
                        max_iter=TRAIN_EPOCHS, alpha=0.0, random_state=s)
    clf.fit(X_train, Y_train)
    sk_same_accs.append(clf.score(X_val, Y_val))

print("sklearn  (same hp):", [round(a, 4) for a in sk_same_accs],
      " mean = {:.4f}, std = {:.4f}".format(np.mean(sk_same_accs), np.std(sk_same_accs, ddof=1)))
print("pytorch  (same hp):", [round(a, 4) for a in PT_SEED_ACCS],
      " mean = {:.4f}, std = {:.4f}".format(np.mean(PT_SEED_ACCS), np.std(PT_SEED_ACCS, ddof=1)))

# H0: the mean validation accuracy of the two implementations is the same.
t_stat, p_val = stats.ttest_ind(sk_same_accs, PT_SEED_ACCS, equal_var=False)   # Welch's t-test
print("\nWelch two-sample t-test: t = {:.3f}, p = {:.4f}".format(t_stat, p_val))

# For reference only (weaker pairing argument, see text above)
t_p, p_p = stats.ttest_rel(sk_same_accs, PT_SEED_ACCS)
print("Paired t-test (reference):  t = {:.3f}, p = {:.4f}".format(t_p, p_p))

alpha_level = 0.05
if p_val < alpha_level:
    print(f"\np < {alpha_level}: reject H0 -- the two implementations give significantly different accuracies.")
else:
    print(f"\np >= {alpha_level}: fail to reject H0 -- no significant difference between the two implementations "
          "at the 5% level (with n=5 per group the test has low power, so this is not evidence of equality).")

**Interpretation (fill in the numbers after running).** State H0, the test statistic, the p-value and the decision at α = 0.05. With only five runs per group the test has very little power, so *fail to reject* should be phrased as "no evidence of a difference", not "the two are the same". Any residual difference is expected to come from implementation details (sklearn's `adam` uses `beta`/`epsilon` defaults and an internal validation split only when `early_stopping=True`; PyTorch shuffles per epoch; different initialisation schemes).